### Sprint Goal
> Use the cleaned and categorised data to identify the single biggest market opportunity and produce a clear, data-backed product recommendation.
>
> This sprint answers the client's core question:
> **'Where is the Blue Ocean in the snack aisle?'**

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Create master_data folder if not already there
os.makedirs('../Master-Data', exist_ok=True)

INPUT_FILE = '../Master-Data/categorized_food_data.csv'
df = pd.read_csv(INPUT_FILE, low_memory=False)

print(f'   Rows       : {len(df):,}')
print(f'   Categories : {df["Primary_Category"].unique().tolist()}')
df.head(3)

   Rows       : 47,495
   Categories : ['Other Snacks', 'Nuts, Seeds & Dried Fruit', 'Crackers & Savoury Biscuits', 'Protein Bars & Supplements', 'Yogurt & Dairy Snacks', 'Ice Cream & Frozen Snacks', 'Chocolate & Candy', 'Cakes & Pastries', 'Biscuits & Cookies', 'Chips & Crisps', 'Sweet Spreads & Dips', 'Granola, Bars & Cereals', 'Meat & Fish Snacks', 'Popcorn', 'Cheese Snacks', 'Fruit Snacks']


,product_name,categories_tags,ingredients_text,nutriscore_score,energy_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,Primary_Category
0,Pinto Bean,en:asian-style-ready-meal,NaN,NaN,9.0,10.2,3.6,22.5,4.9,3.8,17.5,NaN,Other Snacks
1,pasta,"en:beverages-and-beverages-preparations,en:bev...",NaN,5.0,697.1,6.4,1.3,20.0,1.9,0.8,6.7,0.0009,Other Snacks
2,Eirn original curry Sauce,"en:plant-based-foods-and-beverages,en:plant-ba...",Wheat Flour Sugar Vegetable Fat D Curry Powder...,NaN,1724.0,18.0,8.2,57.2,29.6,1.2,5.2,4.7500,Other Snacks


--
## Define the Healthy Zone

**Business Definition:**
A product is in the **Healthy Zone** (Blue Ocean) if it meets BOTH conditions:
- Protein >= 15g per 100g — high enough to qualify as a protein snack
- Sugar <= 10g per 100g — low enough to avoid the 'sugary snack' label


In [2]:
PROTEIN_THRESHOLD = 15   # g per 100g — minimum for 'high protein'
SUGAR_THRESHOLD   = 10   # g per 100g — maximum for 'low sugar'


df['In_Healthy_Zone'] = (
    (df['proteins_100g'] >= PROTEIN_THRESHOLD) &
    (df['sugars_100g']   <= SUGAR_THRESHOLD)
)

df['In_Danger_Zone'] = (
    (df['sugars_100g']   > 20) &
    (df['proteins_100g'] < 10)
)

total_healthy = df['In_Healthy_Zone'].sum()
total_danger  = df['In_Danger_Zone'].sum()


print(f'\n   Healthy Zone threshold : Protein >= {PROTEIN_THRESHOLD}g AND Sugar <= {SUGAR_THRESHOLD}g')
print(f'   Danger Zone threshold  : Sugar > 20g AND Protein < 10g')
print(f'\n   Products in Healthy Zone : {total_healthy:,} ({total_healthy/len(df)*100:.1f}%)')
print(f'   Products in Danger Zone  : {total_danger:,} ({total_danger/len(df)*100:.1f}%)')
print(f'   Products in neither zone : {len(df)-total_healthy-total_danger:,} ({(len(df)-total_healthy-total_danger)/len(df)*100:.1f}%)')


   Healthy Zone threshold : Protein >= 15g AND Sugar <= 10g
   Danger Zone threshold  : Sugar > 20g AND Protein < 10g

   Products in Healthy Zone : 5,420 (11.4%)
   Products in Danger Zone  : 8,355 (17.6%)
   Products in neither zone : 33,720 (71.0%)


## TASK-27 — Calculate Gap Index Per Category

**Gap Index** = % of products in each category that are already in the Healthy Zone.
**Low % = big gap = big opportunity.**

In [3]:
gap_results = []

for cat in sorted(df['Primary_Category'].unique()):
    cat_df         = df[df['Primary_Category'] == cat]
    total          = len(cat_df)
    in_healthy     = cat_df['In_Healthy_Zone'].sum()
    in_danger      = cat_df['In_Danger_Zone'].sum()
    pct_healthy    = round(in_healthy / total * 100, 2)
    pct_danger     = round(in_danger  / total * 100, 2)
    avg_sugar      = round(cat_df['sugars_100g'].median(), 1)
    avg_protein    = round(cat_df['proteins_100g'].median(), 1)

    gap_results.append({
        'Category'              : cat,
        'Total Products'        : total,
        'In Healthy Zone'       : in_healthy,
        '% Healthy'             : pct_healthy,
        'In Danger Zone'        : in_danger,
        '% Danger'              : pct_danger,
        'Median Sugar (g)'      : avg_sugar,
        'Median Protein (g)'    : avg_protein,
    })

gap_df = pd.DataFrame(gap_results).sort_values('% Healthy', ascending=True)

print('\n=== GAP INDEX TABLE (sorted by opportunity size) ===')
print(f'{"Category":<35} {"Total":>7}  {"% Healthy":>10}  {"% Danger":>10}  {"Med Sugar":>10}  {"Med Protein":>12}')
print('-' * 95)
for _, row in gap_df.iterrows():
    flag = ' ← BIGGEST GAP' if row['% Healthy'] == gap_df['% Healthy'].min() else ''
    print(f"{row['Category']:<35} {int(row['Total Products']):>7,}  {row['% Healthy']:>9.1f}%  {row['% Danger']:>9.1f}%  {row['Median Sugar (g)']:>9.1f}g  {row['Median Protein (g)']:>10.1f}g{flag}")


=== GAP INDEX TABLE (sorted by opportunity size) ===
Category                              Total   % Healthy    % Danger   Med Sugar   Med Protein
-----------------------------------------------------------------------------------------------
Fruit Snacks                             14        0.0%       92.9%       45.5g         1.4g ← BIGGEST GAP
Ice Cream & Frozen Snacks               671        0.0%       41.4%       18.9g         3.0g ← BIGGEST GAP
Popcorn                                 123        0.0%       18.7%        1.2g         7.3g ← BIGGEST GAP
Biscuits & Cookies                    1,776        1.1%       71.8%       29.6g         5.3g
Yogurt & Dairy Snacks                 3,924        1.5%        2.4%        9.9g         4.6g
Crackers & Savoury Biscuits           1,071        1.8%       34.5%        8.0g         6.7g
Chips & Crisps                        1,379        1.9%        3.5%        2.4g         6.0g
Chocolate & Candy                     1,330        2.0%       7

## Opportunity category

In [11]:
# Category with the LOWEST % in healthy zone = biggest gap
opportunity_row = gap_df.iloc[0]   # already sorted ascending
opportunity_cat = opportunity_row['Category']
opportunity_pct = opportunity_row['% Healthy']
opportunity_total = int(opportunity_row['Total Products'])

# How many healthy products exist in this category?
healthy_in_cat = int(opportunity_row['In Healthy Zone'])

print(f'   Opportunity Category : {opportunity_cat}')
print(f'   Total products       : {opportunity_total:,}')
print(f'   In Healthy Zone      : {healthy_in_cat:,} ({opportunity_pct}%)')
print(f'   Gap size             : {100 - opportunity_pct:.1f}% of products are NOT in the healthy zone')
print()
print('   This means out of every 100 products in this category,')
print(f'   only {opportunity_pct:.2f} meet both the high-protein and low-sugar criteria.')

   Opportunity Category : Fruit Snacks
   Total products       : 14
   In Healthy Zone      : 0 (0.0%)
   Gap size             : 100.0% of products are NOT in the healthy zone

   This means out of every 100 products in this category,
   only 0.00 meet both the high-protein and low-sugar criteria.


## Protein and Sugar Targets

In [5]:
#  Products already in the healthy zone in the opportunity cat
healthy_products = df[
    (df['Primary_Category'] == opportunity_cat) &
    (df['In_Healthy_Zone']  == True)
].copy()

print(f'Healthy products found in {opportunity_cat}: {len(healthy_products):,}')

if len(healthy_products) > 0:
    ideal_protein   = round(healthy_products['proteins_100g'].median(), 1)
    ideal_sugar     = round(healthy_products['sugars_100g'].median(),   1)
    ideal_fat       = round(healthy_products['fat_100g'].median(),      1)
    ideal_fiber     = round(healthy_products['fiber_100g'].median(),    1)
    ideal_energy    = round(healthy_products['energy_100g'].median(),   0)
else:
    # Fallback to threshold values if no healthy products exist
    ideal_protein   = float(PROTEIN_THRESHOLD)
    ideal_sugar     = float(SUGAR_THRESHOLD)
    ideal_fat       = 10.0
    ideal_fiber     = 5.0
    ideal_energy    = 300.0

print('=== IDEAL NUTRITIONAL PROFILE FOR NEW PRODUCT ===')
print(f'   Based on median values of existing healthy products in {opportunity_cat}')
print()
print(f'   Protein : {ideal_protein}g per 100g')
print(f'   Sugar   : {ideal_sugar}g per 100g')
print(f'   Fat     : {ideal_fat}g per 100g')
print(f'   Fiber   : {ideal_fiber}g per 100g')
print(f'   Energy  : {ideal_energy:.0f} kcal per 100g')

Healthy products found in Fruit Snacks: 0
=== IDEAL NUTRITIONAL PROFILE FOR NEW PRODUCT ===
   Based on median values of existing healthy products in Fruit Snacks

   Protein : 15.0g per 100g
   Sugar   : 10.0g per 100g
   Fat     : 10.0g per 100g
   Fiber   : 5.0g per 100g
   Energy  : 300 kcal per 100g


## Key Insights

In [6]:
# ── Generate the required insight sentence ────────────────────
insight_sentence = (
    f"Based on the data, the biggest market opportunity is in "
    f"{opportunity_cat}, specifically targeting products with "
    f"{ideal_protein:.0f}g of protein and less than "
    f"{ideal_sugar:.0f}g of sugar per 100g."
)

supporting_context = (
    f"Only {opportunity_pct:.1f}% of the {opportunity_total:,} products "
    f"currently in the {opportunity_cat} category occupy the "
    f"high-protein / low-sugar quadrant. This is the lowest "
    f"penetration of any snack category analysed, signalling "
    f"clear unmet consumer demand."
)

print('╔══════════════════════════════════════════════════════════════════╗')
print('║              KEY INSIGHT — COPY TO POWER BI DASHBOARD           ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print()
print(f'  {insight_sentence}')
print()
print('╠══════════════════════════════════════════════════════════════════╣')
print('║              SUPPORTING CONTEXT                                  ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print()
print(f'  {supporting_context}')
print()
print('╚══════════════════════════════════════════════════════════════════╝')
print()

╔══════════════════════════════════════════════════════════════════╗
║              KEY INSIGHT — COPY TO POWER BI DASHBOARD           ║
╠══════════════════════════════════════════════════════════════════╣

  Based on the data, the biggest market opportunity is in Fruit Snacks, specifically targeting products with 15g of protein and less than 10g of sugar per 100g.

╠══════════════════════════════════════════════════════════════════╣
║              SUPPORTING CONTEXT                                  ║
╠══════════════════════════════════════════════════════════════════╣

  Only 0.0% of the 14 products currently in the Fruit Snacks category occupy the high-protein / low-sugar quadrant. This is the lowest penetration of any snack category analysed, signalling clear unmet consumer demand.

╚══════════════════════════════════════════════════════════════════╝

 Key Insight sentence generated
   --> Copy the sentence above into your Power BI dashboard text box


## Bonus: The Hidden Gem (Story 5)

Identify the **Top 3 protein sources** driving high protein content in the healthy products.
These become the R&D team's ingredient recommendations.

In [7]:
from collections import Counter

# High protein products with ingredients data
hp_with_ingredients = df[
    (df['proteins_100g']    >= PROTEIN_THRESHOLD) &
    (df['ingredients_text'].notna())
].copy()

print(f'High-protein products with ingredients text: {len(hp_with_ingredients):,}')

#Protein source keywords
PROTEIN_SOURCES = [
    'whey', 'peanut', 'soy', 'soya', 'almond', 'chickpea',
    'lentil', 'egg', 'milk protein', 'casein', 'hemp',
    'pea protein', 'rice protein', 'quinoa',
    'oat', 'chia', 'sesame', 'sunflower seed'
]

#  Count occurrences
source_counts = Counter()
for text in hp_with_ingredients['ingredients_text'].str.lower():
    for source in PROTEIN_SOURCES:
        if source in str(text):
            source_counts[source] += 1

top_sources = pd.DataFrame(
    source_counts.most_common(10),
    columns=['Ingredient', 'Product Count']
)
top_sources['Rank'] = range(1, len(top_sources) + 1)
top_sources['% of High-Protein Products'] = (
    top_sources['Product Count'] / len(hp_with_ingredients) * 100
).round(1)

print()
print('=== TOP PROTEIN SOURCES IN HIGH-PROTEIN SNACKS ===')
print()
medals = ['1st', '2nd', '3rd']
for i, row in top_sources.head(10).iterrows():
    rank  = medals[i] if i < 3 else f'   {i+1}th'
    print(f"  {rank}  {row['Ingredient'].title():<20}  {int(row['Product Count']):>6,} products  ({row['% of High-Protein Products']:.1f}%)")

print()
top3 = top_sources.head(3)['Ingredient'].str.title().tolist()
print(f'  R&D RECOMMENDATION: Use {top3[0]}, {top3[1]}, or {top3[2]} as primary protein sources.')
print()
print( 'Top 3 protein sources identified')

High-protein products with ingredients text: 4,076

=== TOP PROTEIN SOURCES IN HIGH-PROTEIN SNACKS ===

  1st  Soy                      815 products  (20.0%)
  2nd  Peanut                   468 products  (11.5%)
  3rd  Oat                      302 products  (7.4%)
     4th  Almond                   277 products  (6.8%)
     5th  Whey                     230 products  (5.6%)
     6th  Sesame                   149 products  (3.7%)
     7th  Egg                      130 products  (3.2%)
     8th  Soya                     130 products  (3.2%)
     9th  Sunflower Seed            96 products  (2.4%)
     10th  Milk Protein              87 products  (2.1%)

  R&D RECOMMENDATION: Use Soy, Peanut, or Oat as primary protein sources.

Top 3 protein sources identified


In [8]:
#  Export Gap Index
gap_df.to_csv('../Master-Data/powerbi_gap_index.csv', index=False)
print('Exported: Master-Data/powerbi_gap_index.csv')

# Export Protein Sources
top_sources.to_csv('../Master-Data/powerbi_protein_sources.csv', index=False)
print('Exported: Master-Data/powerbi_protein_sources.csv')



Exported: Master-Data/powerbi_gap_index.csv
Exported: Master-Data/powerbi_protein_sources.csv
